In [4]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder


In [5]:
df = pd.read_csv('covid_toy.csv')

In [6]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [23]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [7]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)

In [9]:
X_train

,age,gender,fever,cough,city
3,31,Female,98.0,Mild,Kolkata
21,73,Male,98.0,Mild,Bangalore
13,64,Male,102.0,Mild,Bangalore
2,42,Male,101.0,Mild,Delhi
42,27,Male,100.0,Mild,Delhi
...,...,...,...,...,...
37,55,Male,100.0,Mild,Kolkata
71,75,Female,104.0,Strong,Delhi
75,5,Male,102.0,Mild,Kolkata
20,12,Male,98.0,Strong,Bangalore


<h2>normal methods</h2>

In [10]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.transform(X_test[['fever']])
                                 
X_train_fever.shape

(80, 1)

In [11]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [13]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    drop='first',
    sparse_output=False,
    handle_unknown='ignore'
)

# Fit on training data
X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])

# Transform test data
X_test_gender_city = ohe.transform(X_test[['gender', 'city']])

X_train_gender_city.shape


(80, 4)

In [14]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape


(80, 1)

In [15]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

<h2>Mentos Zindagi</h2>

In [16]:
from sklearn.compose import ColumnTransformer

<b>The transformers argument</b>

    transformers=[
    -->a list
    -->Each element defines :
        (name, transformer_object, columns)

<b>strategy='most_frequent'</b>
    Replaces missing values with:
        the most common value in that column


In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer

transformer = ColumnTransformer(
    transformers=[
        ('tnf1', SimpleImputer(strategy='most_frequent'), ['fever']),
        ('tnf2', OrdinalEncoder(categories=[['Mild','Strong']]), ['cough']),
        ('tnf3', OneHotEncoder(
            drop='first',
            sparse_output=False,
            handle_unknown='ignore'
        ), ['gender','city'])
    ],
    remainder='passthrough'
)

In [21]:
transformer.fit_transform(X_train).shape

(80, 7)

In [22]:
transformer.transform(X_test).shape

(20, 7)